In [ ]:
import os
import cv2
import numpy as np
import torch
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"

script_dir = os.path.dirname(os.path.abspath(os.getcwd()))
segmentation_model = YOLO("yolo26n-seg.pt")
depth_model = YOLO("yolo26n-depth.pt")

def Webcam(cameras : int):
    if type(cameras) == int and cameras:
        return cv2.VideoCapture(cameras)
    
camera = Webcam(0)

if not camera.isOpened():
    raise RuntimeError("Failed to Capture Webcam")

camera.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

cv2.namedWindow(
    "Hand Distance Measurement",
    cv2.WINDOW_NORMAL
)

while True:

    success, frame = camera.read()

    if not success:
        break

    frame = cv2.flip(frame, 1)

    detection_result = segmentation_model.predict(
        frame,
        conf=0.35,
        imgsz=640,
        retina_masks=True,
        device=DEVICE,
        verbose=False
    )[0]

    depth_result = depth_model.predict(
        frame,
        imgsz=768,
        device=DEVICE,
        verbose=False
    )[0]

    depth_map = (
        depth_result.depth.data
        .cpu()
        .numpy()
        .squeeze()
    )

    if detection_result.masks is not None:

        masks = (
            detection_result.masks.data
            .cpu()
            .numpy()
        )

        boxes = detection_result.boxes

        for index, box in enumerate(boxes):

            if index >= len(masks):
                break

            class_id = int(box.cls[0].item())

            class_name = detection_result.names[class_id]

            x1, y1, x2, y2 = map(
                int,
                box.xyxy[0].cpu().numpy()
            )

            object_mask = masks[index] > 0.5

            if object_mask.shape != depth_map.shape:

                object_mask = cv2.resize(
                    object_mask.astype(np.uint8),
                    (depth_map.shape[1], depth_map.shape[0]),
                    interpolation=cv2.INTER_NEAREST
                ).astype(bool)

            valid_pixels = (
                object_mask
                & np.isfinite(depth_map)
                & (depth_map > 0)
            )

            if np.any(valid_pixels):

                distance = float(
                    np.median(depth_map[valid_pixels])
                )

                distance_label = f"{distance:.2f} m"

            else:
                distance_label = "N/A"

            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )

            contours, _ = cv2.findContours(
                object_mask.astype(np.uint8),
                cv2.RETR_EXTERNAL,
                cv2.CHAIN_APPROX_SIMPLE
            )

            cv2.drawContours(
                frame,
                contours,
                -1,
                (255, 0, 0),
                2
            )

            label = (
                f"Class = {class_name} | "
                f"Distance = {distance_label}"
            )

            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 0.5
            thickness = 2

            text_size, baseline = cv2.getTextSize(
                label,
                font,
                font_scale,
                thickness
            )

            text_width, text_height = text_size

            label_x = max(
                0,
                min(x1, frame.shape[1] - text_width - 10)
            )

            label_y = max(
                text_height + 12,
                y1 - 10
            )

            cv2.rectangle(
                frame,
                (label_x, label_y - text_height - 8),
                (label_x + text_width + 8, label_y + baseline),
                (0, 255, 0),
                -1
            )

            cv2.putText(
                frame,
                label,
                (label_x + 4, label_y - 4),
                font,
                font_scale,
                (0, 0, 0),
                thickness
            )

    cv2.imshow(
        "Hand Distance Measurement",
        frame
    )

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

camera.release()
cv2.destroyAllWindows()